# POLAR: Detecting Multilingual, Multicultural and Multievent Online Polarization
## SemEval-2026 Task 9 - Complete Multi-Task Pipeline

This notebook implements an end-to-end multi-task learning pipeline for **all three subtasks** of the SemEval-2026 POLAR shared task:
- **Subtask 1: Polarization Detection** (Binary: `polarization`)
- **Subtask 2: Polarization Type Classification** (Multi-Label: `political`, `racial/ethnic`, `religious`, `gender/sexual`, `other`)
- **Subtask 3: Polarization Manifestation Identification** (Multi-Label: `stereotype`, `vilification`, `dehumanization`, `extreme_language`, `lack_of_empathy`, `invalidation`)

The dataset spans **22 languages**: Amharic, Arabic, Bengali, Burmese, Chinese, English, German, Hausa, Hindi, Italian, Khmer, Nepali, Odia, Persian, Polish, Punjabi, Russian, Spanish, Swahili, Telugu, Turkish, Urdu.


In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent))

from src.config import (
    LANGUAGES, LANGUAGE_NAMES, TRAIN_DIR, DEV_DIR, TEST_DIR,
    SUBTASK_1_COLS, SUBTASK_2_COLS, SUBTASK_3_COLS, SUBTASK_3_EXCLUDED_LANGS
)
from src.dataset import load_polar_split, PolarMultiTaskDataset
from src.utils import compute_all_metrics
from src.models import PolarMultiTaskModel

# Configure visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print(f"PyTorch Version: {torch.__version__}")
print(f"MPS Available:   {torch.backends.mps.is_available()}")


## 1. Exploratory Data Analysis (EDA)
Let's inspect dataset volume, language representation, and label distributions across the three subtasks.


In [ ]:
# Load full training and validation sets
train_df = load_polar_split("train")
dev_df = load_polar_split("dev")

print(f"Total Train Samples: {len(train_df):,}")
print(f"Total Dev Samples:   {len(dev_df):,}")
print(f"Number of Languages: {len(train_df['lang'].unique())}")
train_df.head()


In [ ]:
# Samples per language in Train set
lang_counts = train_df["lang"].map(LANGUAGE_NAMES).value_counts()

plt.figure(figsize=(14, 5))
sns.barplot(x=lang_counts.index, y=lang_counts.values, palette="crest")
plt.title("Sample Distribution Across 22 Languages (Train Set)", fontsize=14, fontweight="bold")
plt.xlabel("Language")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Subtask 1, 2, and 3 Prevalence
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subtask 1
s1_counts = train_df["polarization"].value_counts(normalize=True) * 100
axes[0].pie(s1_counts, labels=["Non-Polarized (0)", "Polarized (1)"], autopct='%1.1f%%', colors=["#4C72B0", "#DD8452"], startangle=90)
axes[0].set_title("Subtask 1: Polarization Detection", fontweight="bold")

# Subtask 2
s2_counts = train_df[SUBTASK_2_COLS].mean().sort_values(ascending=False) * 100
sns.barplot(x=s2_counts.values, y=s2_counts.index, ax=axes[1], palette="flare")
axes[1].set_title("Subtask 2: Polarization Types (%)", fontweight="bold")
axes[1].set_xlabel("% Positive Rate")

# Subtask 3
s3_valid_df = train_df[~train_df["lang"].isin(SUBTASK_3_EXCLUDED_LANGS)]
s3_counts = s3_valid_df[SUBTASK_3_COLS].mean().sort_values(ascending=False) * 100
sns.barplot(x=s3_counts.values, y=s3_counts.index, ax=axes[2], palette="magma")
axes[2].set_title("Subtask 3: Manifestations (%)", fontweight="bold")
axes[2].set_xlabel("% Positive Rate")

plt.tight_layout()
plt.show()


## 2. Multilingual TF-IDF Baseline (Word + Char N-grams)
We train a linear baseline with balanced class weighting to establish strong benchmark numbers for all 3 subtasks.


In [ ]:
from src.baseline import train_and_eval_baseline

baseline_results = train_and_eval_baseline()


## 3. Deep Learning: Unified Multi-Task Multilingual Transformer
We use a shared multilingual encoder (`xlm-roberta-base` or `microsoft/mdeberta-v3-base`) with three branched classification heads:
$$\mathcal{L}_{total} = \lambda_1 \mathcal{L}_{	ext{subtask1}} + \lambda_2 \mathcal{L}_{	ext{subtask2}} + \lambda_3 \mathcal{L}_{	ext{subtask3}}$$


In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = PolarMultiTaskModel(model_name=model_name)

print("Unified Model Architecture:")
print(f"• Encoder Backbone: {model_name}")
print(f"• Subtask 1 Head:   {model.head_subtask1}")
print(f"• Subtask 2 Head:   {model.head_subtask2}")
print(f"• Subtask 3 Head:   {model.head_subtask3}")


## 4. Test Inference & Leaderboard Submission
Generate blind test predictions for all 22 languages matching the exact CodaBench submission format.


In [ ]:
from src.predict import predict_split

# Generate predictions for the test split
test_predictions = predict_split(
    model_checkpoint_dir=Path.cwd().parent / "models" / "best_multitask_polar_model",
    split_dir=TEST_DIR
)
test_predictions.head()
